# Stage 2: CyTOF harmonization

Runs the source notebook's SOM-OT correction, per-file CyTOF normalization, GFP-high balancing, marker scaling, and pooled export in order. The two scientific helper imports below use the same module names as the source notebook and are not installed or reimplemented here.

In [1]:
from pathlib import Path
import sys

# Support launching Jupyter from either the checkout root or its notebooks folder.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "app").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
# Prefer this checkout over another installed copy or worktree of app.
sys.path.insert(0, str(PROJECT_ROOT))

# Add the repository's vendored correction and normalization libraries.
sys.path.insert(0, str(PROJECT_ROOT / "external" / "som_ot_warp_repo_anchor"))
sys.path.insert(0, str(PROJECT_ROOT / "external" / "cytof_helper" / "cytof_helper"))
import app  # noqa: E402

# Fail early if Python resolved app from outside this checkout.
if Path(app.__file__).resolve().parent != PROJECT_ROOT / "app":
    raise RuntimeError("app was imported from outside the intended checkout")

In [2]:
from app.dotenv import OUTPUT_ROOT
from somot_warp.multi_batch import MultiBatchSOMOTWarp
from cytof_transform import CytofTransformConfig, cytof_transform_global
from IPython.display import display

from app.harmonization import (
    balance_samples,
    combine_scaled_experiments,
    fit_batch_correction,
    process_batch_correction_files,
    process_normalization_files,
    process_scaling_files,
)

In [3]:
EXPERIMENTS = ["batch2", "batch3", "batch4_extra_treatment"]
PROCESSING_ROOT = OUTPUT_ROOT / "4. data processing after GFP split"
ARCSINH_IN = PROCESSING_ROOT / "ARCSINH_b"
BATCH_CORRECTED_OUT = PROCESSING_ROOT / "BATCH_CORRECTED_c"
NORMALIZED_OUT = PROCESSING_ROOT / "NORMALIZED_d"
BALANCED_OUT = (
    OUTPUT_ROOT
    / "5. GFP_high_balanced_per_experiment, & Z_scored"
    / "GFP_high_balanced_per_experiment"
)
ZSCORED_OUT = (
    OUTPUT_ROOT
    / "5. GFP_high_balanced_per_experiment, & Z_scored"
    / "7. GFP_high_balanced_per_experiment_ZSCORED"
)

In [4]:
print(f"Output directory: {OUTPUT_ROOT}")

Output directory: /Users/yishai/Desktop/epigenetics_field_theory/results/experiment_141026


## 1. SOM-OT batch correction

Fits on both GFP branches, in the source experiment and file order, using the source shared-finite-marker selection, seed-42 shuffling, and 10,000-cell-per-condition cap.

In [5]:
batch_model, correction_markers, invalid_markers, valid_markers, fit_subset = fit_batch_correction(
    ARCSINH_IN,
    EXPERIMENTS,
    model_factory=MultiBatchSOMOTWarp,
)
correction_summary = process_batch_correction_files(
    ARCSINH_IN,
    BATCH_CORRECTED_OUT,
    EXPERIMENTS,
    batch_model,
    correction_markers,
    invalid_markers,
)
print("Markers used:", correction_markers)
print("Excluded markers:", invalid_markers)
display(
    fit_subset.groupby(["_batch", "condition"], sort=False)
    .size()
    .rename("n_fit_cells")
    .reset_index()
)
display(correction_summary)

 [ 8000 / 8000 ] 100% - 0:00:00 left 
 quantization error: 4.185790400317817
 [ 8000 / 8000 ] 100% - 0:00:00 left 
 quantization error: 4.167643611296765


  0%|          | 0/64 [00:00<?, ?it/s]/Users/yishai/Desktop/epigenetics_field_theory/.venv/lib/python3.11/site-packages/ot/unbalanced/_sinkhorn.py:757: RuntimeWarning: divide by zero encountered in divide
  u = (a / Kv) ** fi_1
/Users/yishai/Desktop/epigenetics_field_theory/.venv/lib/python3.11/site-packages/ot/unbalanced/_sinkhorn.py:770: UserWarning: Numerical errors at iteration 0
  warnings.warn("Numerical errors at iteration %s" % i)
/Users/yishai/Desktop/epigenetics_field_theory/.venv/lib/python3.11/site-packages/ot/unbalanced/_sinkhorn.py:757: RuntimeWarning: divide by zero encountered in divide
  u = (a / Kv) ** fi_1
/Users/yishai/Desktop/epigenetics_field_theory/.venv/lib/python3.11/site-packages/ot/unbalanced/_sinkhorn.py:770: UserWarning: Numerical errors at iteration 0
  warnings.warn("Numerical errors at iteration %s" % i)
  2%|▏         | 1/64 [00:00<00:14,  4.31it/s]/Users/yishai/Desktop/epigenetics_field_theory/.venv/lib/python3.11/site-packages/ot/unbalanced/_sinkhorn.

Markers used: ['CD109', 'CD24', 'CD44', 'CD45', 'E-cadherin', 'EZH2', 'EpCAM', 'GAL4', 'GATA6', 'GFP', 'H2AK119ub', 'H3', 'H3.3', 'H3K27ac', 'H3K27me3', 'H3K36me2', 'H3K36me3', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9ac', 'H3K9me2', 'H3K9me3', 'H4', 'H4K16ac', 'H4K20me3', 'KI67', 'KRT17', 'KRT5', 'MBD', 'Pan-KRT', 'TSPAN8', 'Vimentin', 'pH2A.X', 'pRb']
Excluded markers: ['ACSS2', 'CD133', 'CXCR4', 'H2BK5ac', 'H3S28p', 'MHC-II', 'S100A4', 'SLC44A4']


,_batch,condition,n_fit_cells
0,batch2,treated,10000
1,batch2,untreated,10000
2,batch3,treated,10000
3,batch3,untreated,10000
4,batch4_extra_treatment,treated,10000
5,batch4_extra_treatment,untreated,10000


,input_file,output_file,n_cells,n_markers_corrected
0,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,1638,35
1,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,681,35
2,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,1923,35
3,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,3399,35
4,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,4015,35
5,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,2984,35
6,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,3468,35
7,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,4569,35
8,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,3155,35
9,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,4997,35


## 2. Per-file normalization

In [6]:
normalization_summary, normalization_input_table = process_normalization_files(
    BATCH_CORRECTED_OUT,
    NORMALIZED_OUT,
    CytofTransformConfig,
    cytof_transform_global,
)
display(normalization_input_table)
display(normalization_summary)

[CyTOF-transform] Computed PC1 technical factor for 'global' (explained var PC1 = 0.761)
[CyTOF-transform] Computed PC1 technical factor for 'global' (explained var PC1 = 0.815)
[CyTOF-transform] Computed PC1 technical factor for 'global' (explained var PC1 = 0.817)
[CyTOF-transform] Computed PC1 technical factor for 'global' (explained var PC1 = 0.849)
[CyTOF-transform] Computed PC1 technical factor for 'global' (explained var PC1 = 0.836)
[CyTOF-transform] Computed PC1 technical factor for 'global' (explained var PC1 = 0.814)
[CyTOF-transform] Computed PC1 technical factor for 'global' (explained var PC1 = 0.823)
[CyTOF-transform] Computed PC1 technical factor for 'global' (explained var PC1 = 0.806)
[CyTOF-transform] Computed PC1 technical factor for 'global' (explained var PC1 = 0.804)
[CyTOF-transform] Computed PC1 technical factor for 'global' (explained var PC1 = 0.801)
[CyTOF-transform] Computed PC1 technical factor for 'global' (explained var PC1 = 0.814)
[CyTOF-transform] Com

,experiment,condition,sample_name,gfp_group,input_file,control_markers,n_markers_to_correct,markers_to_correct
0,batch2,treated,batch2_1473_gem,GFP_high,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,25,EZH2 | GAL4 | GATA6 | H2AK119ub | H3K27ac | H3...
1,batch2,treated,batch2_1484_gem,GFP_high,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,25,EZH2 | GAL4 | GATA6 | H2AK119ub | H3K27ac | H3...
2,batch2,treated,batch2_1494_gem,GFP_high,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,25,EZH2 | GAL4 | GATA6 | H2AK119ub | H3K27ac | H3...
3,batch2,untreated,batch2_1474_unt,GFP_high,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,25,EZH2 | GAL4 | GATA6 | H2AK119ub | H3K27ac | H3...
4,batch2,untreated,batch2_1485_unt,GFP_high,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,25,EZH2 | GAL4 | GATA6 | H2AK119ub | H3K27ac | H3...
5,batch2,untreated,batch2_1610_unt,GFP_high,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,25,EZH2 | GAL4 | GATA6 | H2AK119ub | H3K27ac | H3...
6,batch2,treated,batch2_1473_gem,GFP_low,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,25,EZH2 | GAL4 | GATA6 | H2AK119ub | H3K27ac | H3...
7,batch2,treated,batch2_1484_gem,GFP_low,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,25,EZH2 | GAL4 | GATA6 | H2AK119ub | H3K27ac | H3...
8,batch2,treated,batch2_1494_gem,GFP_low,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,25,EZH2 | GAL4 | GATA6 | H2AK119ub | H3K27ac | H3...
9,batch2,untreated,batch2_1474_unt,GFP_low,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,25,EZH2 | GAL4 | GATA6 | H2AK119ub | H3K27ac | H3...


,experiment,condition,sample_name,gfp_group,input_arcsinh_file,output_normalized_file,control_markers,n_cells,n_markers_corrected,markers_corrected
0,batch2,treated,batch2_1473_gem,GFP_high,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,1638,25,EZH2 | GAL4 | GATA6 | H2AK119ub | H3K27ac | H3...
1,batch2,treated,batch2_1484_gem,GFP_high,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,681,25,EZH2 | GAL4 | GATA6 | H2AK119ub | H3K27ac | H3...
2,batch2,treated,batch2_1494_gem,GFP_high,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,1923,25,EZH2 | GAL4 | GATA6 | H2AK119ub | H3K27ac | H3...
3,batch2,untreated,batch2_1474_unt,GFP_high,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,3399,25,EZH2 | GAL4 | GATA6 | H2AK119ub | H3K27ac | H3...
4,batch2,untreated,batch2_1485_unt,GFP_high,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,4015,25,EZH2 | GAL4 | GATA6 | H2AK119ub | H3K27ac | H3...
5,batch2,untreated,batch2_1610_unt,GFP_high,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,2984,25,EZH2 | GAL4 | GATA6 | H2AK119ub | H3K27ac | H3...
6,batch2,treated,batch2_1473_gem,GFP_low,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,3468,25,EZH2 | GAL4 | GATA6 | H2AK119ub | H3K27ac | H3...
7,batch2,treated,batch2_1484_gem,GFP_low,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,4569,25,EZH2 | GAL4 | GATA6 | H2AK119ub | H3K27ac | H3...
8,batch2,treated,batch2_1494_gem,GFP_low,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,3155,25,EZH2 | GAL4 | GATA6 | H2AK119ub | H3K27ac | H3...
9,batch2,untreated,batch2_1474_unt,GFP_low,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,4997,25,EZH2 | GAL4 | GATA6 | H2AK119ub | H3K27ac | H3...


## 3. Balance normalized GFP-high samples

Excludes batch2 sample 1484 only at this stage and uses seed 42 plus the sorted sample position, matching the source cell.

In [7]:
balanced_by_experiment, cell_counts, balance_summary = balance_samples(
    NORMALIZED_OUT, BALANCED_OUT, EXPERIMENTS
)
display(cell_counts.sort_values(["experiment", "condition", "sample_name"]))
display(balance_summary.sort_values(["experiment", "condition", "sample_name"]))

,experiment,sample_name,condition,file,n_GFP_high_cells,skipped,skip_reason
0,batch2,batch2_1473_gem,treated,/Users/yishai/Desktop/epigenetics_field_theory...,1638,False,
1,batch2,batch2_1484_gem,treated,/Users/yishai/Desktop/epigenetics_field_theory...,681,True,batch2 sample 1484 skipped: very small GFP+ sa...
2,batch2,batch2_1494_gem,treated,/Users/yishai/Desktop/epigenetics_field_theory...,1923,False,
3,batch2,batch2_1474_unt,untreated,/Users/yishai/Desktop/epigenetics_field_theory...,3399,False,
4,batch2,batch2_1485_unt,untreated,/Users/yishai/Desktop/epigenetics_field_theory...,4015,False,
5,batch2,batch2_1610_unt,untreated,/Users/yishai/Desktop/epigenetics_field_theory...,2984,False,
6,batch3,batch3_1493_gem,treated,/Users/yishai/Desktop/epigenetics_field_theory...,4252,False,
7,batch3,batch3_1496_gem,treated,/Users/yishai/Desktop/epigenetics_field_theory...,13863,False,
8,batch3,batch3_1497_gem,treated,/Users/yishai/Desktop/epigenetics_field_theory...,13712,False,
9,batch3,batch3_1498_gem,treated,/Users/yishai/Desktop/epigenetics_field_theory...,5129,False,


,experiment,sample_name,condition,n_before,n_after_balancing,limiting_n_for_experiment,is_limiting_sample
0,batch2,batch2_1473_gem,treated,1638,1638,1638,True
1,batch2,batch2_1494_gem,treated,1923,1638,1638,False
2,batch2,batch2_1474_unt,untreated,3399,1638,1638,False
3,batch2,batch2_1485_unt,untreated,4015,1638,1638,False
4,batch2,batch2_1610_unt,untreated,2984,1638,1638,False
5,batch3,batch3_1493_gem,treated,4252,4252,4252,True
6,batch3,batch3_1496_gem,treated,13863,4252,4252,False
7,batch3,batch3_1497_gem,treated,13712,4252,4252,False
8,batch3,batch3_1498_gem,treated,5129,4252,4252,False
9,batch3,batch3_1472_unt,untreated,14736,4252,4252,False


## 4. Per-experiment z-scoring and pooled output

Marker columns follow the source cell's fixed list; population standard deviation (`ddof=0`) and its zero-variance-to-zero rule are retained.

In [8]:
zscore_summary, marker_zscore_proof = process_scaling_files(BALANCED_OUT, ZSCORED_OUT, EXPERIMENTS)
pooled_scaled = combine_scaled_experiments(ZSCORED_OUT, EXPERIMENTS)
display(zscore_summary)
display(marker_zscore_proof.head(100))
print("Pooled scaled table shape:", pooled_scaled.shape)

,experiment,input_file,output_file,n_cells,n_markers_zscored,markers_zscored,markers_skipped_no_numeric_data,zero_variance_markers_set_to_0,max_abs_mean_after_zscore,max_abs_sd_minus_1_after_zscore,ready_for_umap
0,batch2,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,8190,37,"CD109, CD24, CD44, CD45, DNA, E-cadherin, EZH2...",H2A,,4.025541e-16,2.220446e-16,True
1,batch3,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,29764,37,"CD109, CD24, CD44, CD45, DNA, E-cadherin, EZH2...",H2A,,7.639218e-16,2.220446e-16,True
2,batch4_extra_treatment,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,19292,38,"CD109, CD24, CD44, CD45, DNA, E-cadherin, EZH2...",,,1.514489e-15,2.220446e-16,True


,experiment,marker,mean_after_zscore,sd_after_zscore_ddof0,n_non_na,zero_variance_marker
0,batch2,CD109,-2.776235e-17,1.0,8190,False
1,batch2,CD24,0.000000e+00,1.0,8190,False
2,batch2,CD44,-2.082177e-17,1.0,8190,False
3,batch2,CD45,-4.684897e-17,1.0,8190,False
4,batch2,DNA,2.220988e-16,1.0,8190,False
...,...,...,...,...,...,...
95,batch4_extra_treatment,H3K4me3,2.180392e-16,1.0,19292,False
96,batch4_extra_treatment,H3K9ac,9.428724e-17,1.0,19292,False
97,batch4_extra_treatment,H3K9me2,-8.471119e-17,1.0,19292,False
98,batch4_extra_treatment,H3K9me3,1.797350e-16,1.0,19292,False


Pooled scaled table shape: (57246, 48)
